# Load and display the Bagpipes vs Prospector comparisons!

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib as mpl
mpl.rcParams["text.usetex"] = True

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import pandas as pd
import os
import h5py
from corner import quantile
from astropy.cosmology import WMAP9 as cosmo

import scipy
import numpy as np
import astropy
import scipy.stats as stats
from scipy.stats import binned_statistic

no_spec = [9517, 9809, 11051, 11451, 12133, 17713, 17984, 20195, 20693, 20720, 21472, 22990]
bl_agn = [12020, 18977]
intermediate_ap = [7136, 1904, 7922, 8469, 10314, 11337, 11420, 11451, 17517, 17669, 18332, 21452]

exclude = no_spec + bl_agn + intermediate_ap

Define necessary helper functions

In [ ]:
from astropy.io import fits
from astropy.table import Table

import bagpipes as pipes

def load_bluejay(ID):
    """ Load BlueJay photometry from the BlueJay catalogue(s)"""

    # Blue Jay catalogue
    bluejay_cat = Table.read("bluejay_phot_cat_v1.4.fits")
    
    # 1. List all available HST/NIRCam bands:
    filters = ['F090W', 'F115W', 'F125W', 'F140W', 'F150W', 'F160W', 
               'F200W', 'F277W', 'F356W', 'F410M', 'F444W', 'F606W', 'F814W']
    
    # 2. Find the correct row using the ID column
    # Use a mask rather than (int(ID) - 1) to be safe against non-sequential IDs
    row = bluejay_cat[bluejay_cat['ID'] == int(ID)]

    if len(row) == 0:
        raise ValueError(f"ID {ID} not found in catalogue.")
    
    # 3. Extract fluxes and errors into lists
    fluxes = []
    flux_errs = []

    for f in filters:
        fluxes.append(row[f + "_flux"][0] * 1e6)
        flux_errs.append(row[f + "_flux_err"][0] * 1e6)


    # MIRI catalogue
    miri_cat = Table.read("Phot_Table_MIRI.fits")
    
    # 1. List all available MIRI bands:
    miri_filters = ['F770W', 'F1000W', 'F1800W', 'F2100W']
    
    # 2. Find the correct row using the ID column
    # Use a mask rather than (int(ID) - 1) to be safe against non-sequential IDs
    row = miri_cat[miri_cat['ID'] == int(ID)]

    if len(row) == 0:
        raise ValueError(f"ID {ID} not found in catalogue.")
    
    # 3. Extract fluxes and errors into lists
    for f in miri_filters:
        fluxes.append(row[f + "_flux"][0] * 1e6)
        flux_errs.append(row[f + "_flux_err"][0] * 1e6)
    
    # Now turn these into a 2D array [N_filters, 2]
    # Bagpipes expects photometry[i, 0] = flux, photometry[i, 1] = error
    photometry = np.c_[fluxes, flux_errs]
    
    # 5. Clean up missing data and enforce SNR limits
    for i in range(len(photometry)):
        # Blow up errors for missing data (NaN or 0 flux)
        if (photometry[i, 0] <= 0.) or (np.isnan(photometry[i, 0])):
            photometry[i, :] = [0., 9.9e99]
            continue # Skip SNR check for bad data
    
    return photometry

# Redshift
def get_zred(galaxy_id):    
    # --- Read Blue Jay catalogue ---
    blue = "/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/BlueJay_sample.txt"
    tbl = Table.read(blue, format="ascii.basic")
    
    row = tbl[tbl['id'] == int(galaxy_id)]
    
    # Make sure that the code doesn't crash if it can't find the ID in the catalogue
    if len(row) == 0:   
        return None, False
    
    z_spec = row['z_spec'][0]
    
    if z_spec is not None and not np.isnan(z_spec):
        return z_spec, True
    else:
        z_phot = row['z_phot'][0]
        return z_phot, False

# Star formation histories
def zred_to_agebins(zred, z_limit_sfh=20.0, nbins_sfh=8):
    tuniv = cosmo.age(zred).value*1e9   # Age of the universe at the observed redshift in years
    #tbinmax = tuniv-cosmo.age(z_limit_sfh).value*1e9 # Maximum age bin edge corresponding to z_limit_sfh
    tbinmax = tuniv*0.95
    # Compute edges in logarithmic space
    log_edges = np.append(np.array([0.0, 6.7, 7.0]), np.linspace(7.0, np.log10(tbinmax), int(nbins_sfh-1))[1:])
    bin_edges = 10**log_edges   # Convert back to linear space
    bin_edges /= 1e6 # ensure that edges are in Myr for Bagpipes
    return bin_edges.tolist()   # return list of age bin edges

# Function to read the Prospector output files

In [ ]:
prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/sourcephotonly_wMIRI/'

def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = glob.glob(os.path.join(prosp_dir, f'*{galaxy_id}*.h5'))[0]
    
    #file = './output_12717_onlyphot_wMIRI_mcmc.h5'
    
    print(f"Loading Prospector output file: {file}")
        
    with h5py.File(file, 'r') as f:
        # Step 1: Extract the model parameters from the file attributes and display in human-readable format
        model_params = pkl.loads(f.attrs['model_params'], encoding='latin1')      
        
        # Step 2: Extract the sampling results (chain and weights)
        chain = f['sampling']['chain'][:]
        lnprob = f['sampling']['lnprobability']
        
        # Step 3: Get maximum likelihood index
        imax = np.argmax(lnprob)
        
        try:
            i, j = np.unravel_index(imax, lnprob.shape)
            theta_best = chain[i, j, :].copy()
        except(ValueError):
            theta_best = chain[imax, :].copy()        
        
        weights = f['sampling']['weights'][:] if 'weights' in f['sampling'] else None
        
        # Extract labels for parameters that were "free" (fitted)
        labels = [p['name'] for p in model_params if p.get('isfree', False)]    
        
        # 4. Build the data structure
        results = {
            'meta': {'labels': labels, 'map_idx': imax, 'weights': weights},
            'params': {}
        }
        
        for i, name in enumerate(labels):
            # Use the flattened chain for statistics
            param_samples = chain[:, i]
            
            q16, q50, q84 = quantile(param_samples, [0.16, 0.5, 0.84], weights=weights)
            
            results['params'][name] = {
                'samples': param_samples,
                'map': theta_best[i],   # Use the value from the best vector directly
                'q50': q50,
                'q84': q84,
                'q16': q16
            }
                
            if name == 'dust2': print(theta_best[i])

        return results

example_res = load_prospector_results(12717, prosp_dir)

#print(example_res['params']['dust2']['map'])

phot_table = './Phot_Table_MIRI.fits'
with fits.open(phot_table) as hdul:
    galaxy_ids = hdul[1].data['ID']

for gid in galaxy_ids:
    example_res = load_prospector_results(gid, prosp_dir)

#with open('./params_MAP_12717.pkl', 'rb') as f:
    #data = pkl.load(f)
    #print(data['dust2'])    

# Load Prospector directly from the output pickle files

In [ ]:
prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/params/'

def load_prospector_results(galaxy_id, prosp_dir):
    """Function to load Prospector result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        prosp_dir (str): The directory containing the Prospector output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = os.path.join(prosp_dir, f'params_MAP_{galaxy_id}.pkl')
    
    #file = './output_12717_onlyphot_wMIRI_mcmc.h5'
    
    print(f"Loading Prospector output file: {file}")
    
    with open(file, 'rb') as f:
        data = pkl.load(f)
    
    return data

example_res = load_prospector_results(12717, prosp_dir)

#print(example_res['params']['dust2']['map'])

with open('./params_MAP_12717.pkl', 'rb') as f:
    data = pkl.load(f)
    print(data['gas_logu'])    

# Function to extract data from the bagpipes output files

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri/'

example_id = 7102

def load_bagpipes_results(galaxy_id, run):
    """Function to load Bagpipes result .h5 files and disect its 
    data structure to be used for easy plotting.

    Args:
        galaxy_id (int): The ID of the galaxy for which to load results
        bagp_dir (str): The directory containing the Bagpipes output files

    Returns:
        dict: A comprehensive dictionary storing the parameters samples, MAP values and quantiles
    """
    
    file = os.path.join(bagp_dir, f'{galaxy_id}.h5')    # Only one file per galaxy
    
    print(f"Loading Bagpipes output file: {file}")
    
    with h5py.File(file, 'r') as results:
        
        # Get redshift of the source
        fit_str = results.attrs['fit_instructions']
        fit = eval(fit_str, {"np": np, "array": np.array})
        zred = fit['redshift']
        
        # Extract the sampling results
        chain = results['samples2d']
        
        # Get maximum likelihood index
        imax = np.argmax(results['lnlike'])
        
        # Manually extracted the labels from the results
        labels = ['dsfr1', 'dsfr2', 'dsfr3', 'dsfr4', 'dsfr5', 'dsfr6', 'logmass', 'logzsol', 'dust2', 
                    'duste_gamma', 'dust_index', 'duste_qpah', 'dust_umin', 'gas_logu']
        
        # Build the data structure
        data = {
            'meta': {'labels': labels, 'map_idx': imax, 'weights': None},
            'params': {}
        }
        
        # Loop through each parameter
        for i, name in enumerate(labels):
            samples = chain[:, i]
            q16, q50, q84 = quantile(samples, [0.16, 0.5, 0.84], weights=None)
            
            if name == 'logzsol':   # Convert metallicity to log
                data['params'][name] = {
                    'samples': np.log10(samples),
                    'map': np.log10(samples[imax]),
                    'q16': np.log10(q16),
                    'q50': np.log10(q50),
                    'q84': np.log10(q84)
                }
            
            else:    
                data['params'][name] = {
                    'samples': samples,
                    'map': samples[imax],
                    'q16': q16,
                    'q50': q50,
                    'q84': q84
                }

        data['params']['zred'] = {'samples': None, 'map': zred, 'q16': zred, 'q50': zred, 'q84': zred}
        
    return data
    
example_res = load_bagpipes_results(21424, run='fesc_with_miri')
print(example_res)


# Compare the outputs!

In [ ]:
bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri'
bagp_res = load_bagpipes_results(7102, bagp_dir)

prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/params/'
prosp_res = load_prospector_results(7102, prosp_dir)

print(bagp_res['meta']['labels'])
#print(prosp_res['meta']['labels'])

#print(bagp_res['params']['zred']['map'])
#print(10**prosp_res['params']['duste_qpah']['map'])

def get_comparison_map():
    """
    Returns a dictionary mapping physical concepts to (Bagpipes_key, Prospector_key).
    Format: 'Conceptual Name': (Bagpipes_Label, Prospector_Label, Scale_Factor)
    """
    return {
        'zred':             ('zred', 'zred'),
        'logmass':          ('logmass', 'logmass'), 
        'metallicity':      ('Z/Zsol', 'logzsol'),
        'Av':               ('Av/mag', 'dust2'),
        'qpah':             ('dust:qpah', 'duste_qpah'),
        'dust_umin':        ('dust:umin', 'duste_umin'),
        'dust_gamma':       ('dust:gamma', 'duste_gamma'),
        'dust_index':       ('n_dust', 'dust_index'),
        'logU':             ('nebular:logU', 'gas_logu')
    }

In [ ]:
import pandas as pd

cmap = get_comparison_map()
gal_results = []

# Step 1: Get all IDs from the MIRI catalogue
miri_table = "./Phot_Table_MIRI.fits"
id_table = Table.read(miri_table)
all_ids = [int(val.decode('utf-8') if isinstance(val, bytes) else val) 
                        for val in id_table['ID']]

bagp_dir = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/pipes/posterior/no_fesc_with_miri'
prosp_dir = '/Users/benjamincollins/Data/Bluejay/Prospector/v2.3/params/'

# Loop over your galaxy IDs
for gal_id in all_ids:
    
    if gal_id in exclude:
        print(f"Skipping galaxy ID {gal_id}...")    # Exclude some sources
        continue
    
    try:
        b_res = load_bagpipes_results(gal_id, bagp_dir)
    except:
        print(f"Skipping galaxy ID {gal_id}...")
        continue
    
    p_res = load_prospector_results(gal_id, prosp_dir)
    
    row = {'id': gal_id}
    for col, (b_lab, p_lab) in cmap.items():   
        
        # Get Bagpipes MAP
        b_raw = b_res['params'][b_lab]['map']
        p_raw = p_res[p_lab]
        
        if col == 'metallicity':
            # Convert Bagpipes to log
            b_val = np.log10(b_raw)
            p_val = p_raw
        elif col == 'Av':
            b_val = b_raw
            p_val = p_raw * 1.086
            if p_val < 0: p_val = 0 # Physical floor
        else:
            b_val = b_raw
            p_val = p_raw
        
        row[f'{col}_bagp'] = b_val
        row[f'{col}_pros'] = p_val
        
    gal_results.append(row)

df = pd.DataFrame(gal_results)
print(df)

# Create summary plot between the two!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation

fig_path = '/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/comparison_plots/no_fesc_with_miri/params.png'

# Define the parameters we want to plot
# Format: (conceptual_name, df_prefix, display_label)
params = [
    ('logmass', r'$\log_{10}(M_*/M_\odot)$'),
    ('metallicity', r'$\log_{10}(Z/Z_\odot)$'),
    ('Av', r'$A_V$ [mag]'),
    ('logU', r'$\log_{10}(U)$'),
    ('qpah', r'$q_{PAH}$ [\%]'),
    ('dust_umin', r'Dust $U_{min}$'),
    ('dust_gamma', r'Dust $\gamma$'),
    ('dust_index', r'$n_{dust}$')
]

n_params = len(params)
fig, axes = plt.subplots(2, n_params//2, figsize=(14, 8))

axes = axes.flatten()

for i, (col, label) in enumerate(params):
    ax = axes[i]
    
    x = df[f'{col}_bagp']
    y = df[f'{col}_pros']
    
    # Calculate limits for the 1:1 line
    all_vals = np.concatenate([x.values, y.samples if hasattr(y, 'samples') else y.values])
    all_vals = all_vals[np.isfinite(all_vals)] # Clean NaNs
    
    if len(all_vals) > 0:
        vmin, vmax = np.min(all_vals), np.max(all_vals)
        # Add a 10% buffer
        pad = (vmax - vmin) * 0.1
        vmin -= pad
        vmax += pad
    else:
        vmin, vmax = 0, 1

    # Plot 1:1 Line
    ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)
    
    # Labels and formatting
    ax.set_title(f'{label}', fontsize=14)
    ax.set_xlabel(f'Bagpipes', fontsize=12)
    ax.set_ylabel(f'Prospector', fontsize=12)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Calculate and display Mean Offset
    if i == 0:
        offset = np.nanmedian(y - x)
        scatter = median_abs_deviation(y - x)
        ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    else:
        corr_coef, p_value = stats.pearsonr(x, y)
        ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


# Plot only masses

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(4, 4))

x = df['logmass_bagp']
y = df['logmass_pros']

# Calculate limits for the 1:1 line
all_vals = np.concatenate([x.values, y.samples if hasattr(y, 'samples') else y.values])
all_vals = all_vals[np.isfinite(all_vals)] # Clean NaNs

if len(all_vals) > 0:
    vmin, vmax = np.min(all_vals), np.max(all_vals)
    # Add a 10% buffer
    pad = (vmax - vmin) * 0.1
    vmin -= pad
    vmax += pad
else:
    vmin, vmax = 0, 1

# Plot 1:1 Line
ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)

# Scatter plot
ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)

# Labels and formatting
plt.title(r'$\log_{10}(M_*/M_\odot)$')
ax.set_xlabel(f'Bagpipes', fontsize=12)
ax.set_ylabel(f'Prospector', fontsize=12)
ax.set_xlim(vmin, vmax)
ax.set_ylim(vmin, vmax)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# Calculate and display Mean Offset
offset = np.nanmedian(y - x)
scatter = median_abs_deviation(y - x)
ax.text(0.05, 0.95, f'$\Delta$: {offset:.2f}\n$\sigma$: {scatter:.2f}', 
        transform=ax.transAxes, verticalalignment='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig('./comparison_plots/no_fesc_with_miri/logmasses.png', dpi=300, bbox_inches='tight')
plt.show()


# Plot only dust attenuation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(4, 4))

x = df['Av_bagp']
y = df['Av_pros']

# Calculate limits for the 1:1 line
all_vals = np.concatenate([x.values, y.samples if hasattr(y, 'samples') else y.values])
all_vals = all_vals[np.isfinite(all_vals)] # Clean NaNs

if len(all_vals) > 0:
    vmin, vmax = np.min(all_vals), np.max(all_vals)
    # Add a 10% buffer
    pad = (vmax - vmin) * 0.1
    vmin -= pad
    vmax += pad
else:
    vmin, vmax = 0, 1

# Plot 1:1 Line
ax.plot([vmin, vmax], [vmin, vmax], color='gray', linestyle='--', alpha=0.7, zorder=1)

# Scatter plot
ax.scatter(x, y, alpha=0.6, color='royalblue', edgecolor='white', s=50, zorder=2)

# Labels and formatting
plt.title(r'$\log_{10}(M_*/M_\odot)$')
ax.set_xlabel(f'Bagpipes', fontsize=12)
ax.set_ylabel(f'Prospector', fontsize=12)
ax.set_xlim(vmin, vmax)
ax.set_ylim(vmin, vmax)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# Calculate and display Mean Offset
corr_coef, p_value = stats.pearsonr(x, y)
ax.text(0.05, 0.95, f'r = {corr_coef:.2f}', 
        transform=ax.transAxes, verticalalignment='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.savefig('./comparison_plots/no_fesc_with_miri/dust.png', dpi=300, bbox_inches='tight')
plt.show()


# Open a random parameter file

In [ ]:
with open('./params_MAP_12717.pkl', 'rb') as f:
    data = pkl.load(f)
    print(data['dust2'])    